# 09 · Evidencias de CD e ISI para el escenario de 72 GS/s (D2.1)

Cada sección respalda una decisión **propuesta** (no se toman decisiones nuevas acá): empieza con la decisión en una línea, sigue con la figura o tabla que la respalda y termina con observaciones cuyos números salen de las celdas.

**Escenario:** DP-16QAM, 60 GBd, 1,2 sps (`Scenario(Rs=60e9, sps=Fraction(6, 5))`, B5.4), f<sub>s</sub> = 72 GS/s, 4 canales reales XI, XQ, YI, YQ en Q1.15. Señal común del notebook 08 (`utils_08.common_signal`, RMS medio 0,25).

**Grilla de CD acumulada** (ps/nm): 0, 5, 10, 17, 25, 50, 75, 100, 150, 200, 300, 400, 600, 800. λ = 1550 nm, D = 17 ps/nm/km (z = CD/D), S = 0; los puntos 200 y 800 se repiten con S = 0,057 ps/nm²/km.

**Criterios:** error del método NRMSE ≤ 1e-2 (aceptación del proyecto) y ≤ 1e-3; SQNR del punto fijo ≥ 60 dB (documento común).

**Medición del error (E3 y E7):** contra `cd_reference` (FFT de toda la señal) con el mismo S, sobre la segunda mitad de la señal (sin 256 muestras por borde), por polarización compleja (XI + jXQ, YI + jYQ), informando la peor de las dos. El FIR (B) y la forma de primer orden (A) se ajustan por mínimos cuadrados sobre la primera mitad. El overlap-save (C) se compara con su retardo conocido V/2, sin búsqueda.

**Qué NO hay acá:** RTL ni punto fijo del hardware; los recursos son **estimaciones por fórmula** o valores externos citados.

**Código:** `notebooks/utils_09.py` (reutiliza `utils_08.py`). Resultados en `docs/golden_model/resultados/09_evidencias_cd_isi/` (`fig/` PNG y SVG, `csv/`, `meta.json`, `informe.html`).

In [ ]:
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import scipy
from IPython.display import Markdown, display

import utils_08 as u8
import utils_09 as u
from emulador import plots, rx
from emulador.cd import cd_R, cd_hbar, cd_overlap_save, cd_reference, cd_transfer
from emulador.config import D_si, S_si
from emulador.metrics import group_delay, nrmse
from emulador.stimuli.pulse import CommSignal

plots.use_style()
plt.rcParams["svg.hashsalt"] = "D2.1"  # ids fijos en los SVG: los diffs no cambian si la figura no cambia
T0 = time.perf_counter()
TIMES = {}
FIG = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100", "violet": "#6250d6",
       "red": "#e34948", "muted": "#7A8499", "ink": "#14213D", "band1": "#cde2fb", "band2": "#fde3d6"}
FAMILY = {"gaussian": FIG["blue"], "bessel": FIG["orange"], "butter": FIG["aqua"]}
M4 = (("gaussian", None), ("bessel", 4), ("butter", 4))
show = lambda rows, cols=None, d=4: display(Markdown(u8.md_table(rows, cols or list(dict.fromkeys(k for r in rows for k in r)), d)))
fmtK = lambda k: f">{u.K_B_MAX}" if k is None else k

SIG = u8.common_signal()
CH = SIG["channels"]
X, Y = CH[0] + 1j * CH[1], CH[2] + 1j * CH[3]
N = CH.shape[1]
TR, TE = u8.split_indices(N, 256)
EDGE = 1024
INNER = slice(EDGE, N - EDGE)
REF = {}  # cache de cd_reference: (CD, S) -> (yX, yY)


def ref(cd, S=0.0):
    if (cd, S) not in REF:
        REF[(cd, S)] = (u.cd_ref(X, cd, S), u.cd_ref(Y, cd, S))
    return REF[(cd, S)]


CASES = [(cd, 0.0) for cd in u.CD_GRID] + [(cd, u.S_ALT) for cd in u.CD_S_REPEAT]
print(u.ESC, "| fs =", u.FS / 1e9, "GS/s | muestras por canal:", N, "| RMS medio:", SIG["rms"].mean().round(6))

## Verificación del overlap-save con V ≠ L/2

El del núcleo (`cd_hbar`, `cd_overlap_save`) fija V = N = L/2. `utils_09.os_hbar(L, V, …)` y `os_apply(x, H̄, V)` generalizan: cada trama toma V muestras de la anterior y L − V nuevas, H̄[k] = H(f<sub>k</sub>)·exp(−j2πk(V/2)/L), se descartan las primeras V muestras de la IFFT y la salida queda retrasada V/2. Tres controles:
1. con V = L/2 coincide con el núcleo;
2. con CD = 0 devuelve x retrasada V/2;
3. contra `cd_reference`, el error baja al subir L y es grande cuando R > V + 1 (a 800 ps/nm, R = 33 y V = 32 con (128, 32)).

In [ ]:
d1 = np.max(np.abs(u.os_apply(X, u.os_hbar(256, 128, 200), 128)
                   - cd_overlap_save(X, cd_hbar(128, u.FS, D_si(u.D_PS), u.LAM, u.z_m(200)))))
d2 = np.max(np.abs(u.os_aligned(X, 256, 64, 0) - X))
chk = []
for cd in (200, 800):
    e = [nrmse(u.os_aligned(X, L, V, cd)[TE], ref(cd)[0][TE]) for L, V in u.OS_PAIRS]
    chk.append({"CD ps/nm": cd, "R": u.cd_R_of(cd), **{f"({L},{V})": v for (L, V), v in zip(u.OS_PAIRS, e)}})
show(chk, d=3)
print(f"1) |os_apply - núcleo| máx (L=256, V=128): {d1:.2e}\n2) |os(CD=0) - x| máx: {d2:.2e}")
assert d1 <= 1e-14 and d2 <= 1e-14
assert all(r["(128,32)"] > r["(256,64)"] > r["(512,128)"] for r in chk)
assert chk[1]["(128,32)"] > 1e-2  # R = 33 > V + 1

## E1 · Rango de distancias

**Decisión que respalda (propuesta):** CD nominal 0 a 200 ps/nm, opcional hasta 800 ps/nm; la CD se configura con D, S, z y λ y se informa también como CD acumulada D·z.

**Fuentes externas** (citadas, no recalculadas):

| Escenario | Alcance | CD acumulada | PMD | Fuente |
|---|---|---|---|---|
| Coherente "lite" dentro del data center | 500 m a 10 km | — | — | [1] |
| OIF 800LR | tramo sin amplificar de hasta 10 km | — | — | [2] |
| 800GBASE-LR1, DP-16QAM, banda C | 10 km | 0 a 200 ps/nm | 5 ps | [3] |
| 800GBASE-ER1 | 40 km | 0 a 800 ps/nm | 10 ps | [3] |
| 400ZR / 800ZR (amplificado) | 80 a 120 km | — | — | fuera del alcance: el proyecto marco apunta a interconexión dentro del data center |

**Rango nominal:** 0 a 200 ps/nm (≈ 0 a 11,8 km con D = 17). **Extensión opcional:** hasta 800 ps/nm (≈ 47 km con D = 17; 40 km con D = 20).

**Referencias**
1. Marvell, "O-band coherent: an idea whose time is nearly here" (blog técnico de Marvell).
2. OIF, *800LR Implementation Agreement*, 2025.
3. E. Maniloff, propuesta de 800GBASE-LR1 y 800GBASE-ER1 coherentes (DP-16QAM, banda C) al grupo de tarea IEEE P802.3dj, noviembre de 2023.

_Fecha de publicación, URL y número de documento de [1] a [3]: a completar por Wanda (no se inventan acá)._

**Verificación numérica.** Con S = 0, H<sub>CD</sub>(f) = exp(jπDλ²zf²/c): la fase depende solo de D·z·λ². Se aplican a X tres pares (D, z) con D·z = 200 ps/nm a 1550 nm, y el mismo D·z a 1310 nm.

In [ ]:
t = time.perf_counter()
LAM2 = 1310e-9
base = cd_reference(X, u.FS, D_si(17), u.LAM, 200 / 17 * 1e3)
e1 = []
for D, z_km in ((17, 200 / 17), (20, 10), (5, 40)):
    y = cd_reference(X, u.FS, D_si(D), u.LAM, z_km * 1e3)
    e1.append({"caso": f"D = {D}, z = {z_km:.4g} km, λ = 1550 nm", "D·z ps/nm": D * z_km,
               "R": u.cd_R_of(D * z_km, D), "NRMSE contra D=17, z=200/17 km": nrmse(y, base)})
y1310 = cd_reference(X, u.FS, D_si(17), LAM2, 200 / 17 * 1e3)
cd_eq = 200 * (LAM2 / u.LAM) ** 2
e1.append({"caso": "D = 17, z = 200/17 km, λ = 1310 nm", "D·z ps/nm": 200.0,
           "R": cd_R(u.FS, D_si(17), LAM2, 200 / 17 * 1e3),
           "NRMSE contra D=17, z=200/17 km": nrmse(y1310, base)})
y_eq = cd_reference(X, u.FS, D_si(17), u.LAM, cd_eq / 17 * 1e3)
e1.append({"caso": f"D = 17, λ = 1550 nm, D·z = 200·(1310/1550)² = {cd_eq:.2f} ps/nm", "D·z ps/nm": cd_eq,
           "R": u.cd_R_of(cd_eq), "NRMSE contra D=17, z=200/17 km": nrmse(y_eq, base),
           "NRMSE contra λ = 1310 nm": nrmse(y_eq, y1310)})
for cd in u.CD_S_REPEAT:
    ys = u.cd_ref(X, cd, u.S_ALT)
    e1.append({"caso": f"S = 0,057 contra S = 0 a {cd} ps/nm (1550 nm)", "D·z ps/nm": cd, "R": u.cd_R_of(cd),
               "NRMSE contra S = 0, mismo D·z": nrmse(ys, u.cd_ref(X, cd))})
show(e1, d=3)
u.save_csv(e1, "T-E1_parametros_cd")
TIMES["E1_s"] = time.perf_counter() - t

**Observaciones.**

- Con S = 0, los tres pares (D, z) con D·z = 200 ps/nm a 1550 nm dan la misma salida hasta el redondeo (NRMSE ≤ 6,2e-16). Para la CD alcanza con D·z, pero λ hace falta: con el mismo D·z a 1310 nm la salida cambia (NRMSE 0,561 contra 1550 nm, R = 5 en vez de 9). A 1550 nm coincide con D·z = 200·(1310/1550)² = 142,86 ps/nm (NRMSE 4,4e-16), porque la fase depende de D·z·λ².
- S también hace falta como parámetro: S = 0,057 ps/nm²/km cambia la salida en NRMSE 6,5e-4 a 200 ps/nm y 2,6e-3 a 800 ps/nm. A 800 ps/nm el cambio supera el criterio de 1e-3 del método.
- Los alcances y las CD acumuladas de la tabla de fuentes son valores externos citados. Las referencias [1] a [3] tienen que completarse con fecha, URL o número de documento.

## E2 · Qué tan fuerte es la CD en el rango

**Respalda:** el tamaño del problema en el rango nominal y en el opcional (insumo de E3 a E5).

- **Efecto relativo** ‖y − x<sub>d</sub>‖/‖x‖, con x<sub>d</sub> la entrada desplazada el retardo entero d (|d| ≤ R) que lo minimiza; peor de X e Y, sin 1024 muestras por borde.
- **R** = 2⌊|D|λ²z/(2cT<sub>s</sub>²)⌋ + 1 (`cd_R`).
- **Respuesta al impulso** |h| = |IFFT(H<sub>CD</sub>)| con una FFT de 512 puntos, centrada, para 17, 200 y 800 ps/nm; las líneas verticales marcan ±(R − 1)/2.

In [ ]:
t = time.perf_counter()
e2 = []
for cd in u.CD_GRID:
    R = u.cd_R_of(cd)
    worst = 0.0
    for x, y in zip((X, Y), ref(cd)):
        best = min(np.linalg.norm(y[INNER] - np.roll(x, d)[INNER]) for d in range(-R, R + 1))
        worst = max(worst, best / np.linalg.norm(x[INNER]))
    e2.append({"CD ps/nm": cd, "z km (D=17)": cd / 17, "R": R, "efecto relativo": worst})
show(e2, d=3)
u.save_csv(e2, "T-E2_fuerza_cd")

fig, axs = plt.subplots(1, 3, figsize=(16, 4.6))
cds = [r["CD ps/nm"] for r in e2]
for ax in axs[:2]:
    ax.axvspan(0, u.CD_NOMINAL, color=FIG["band1"], alpha=0.6, lw=0, label="nominal 0–200")
    ax.axvspan(u.CD_NOMINAL, u.CD_OPTIONAL, color=FIG["band2"], alpha=0.6, lw=0, label="opcional 200–800")
    ax.set_xlabel("CD acumulada [ps/nm]"); ax.set_xlim(0, 820)
axs[0].plot(cds, [r["efecto relativo"] for r in e2], "o-", color=FIG["blue"], lw=1.8)
axs[0].set_ylabel("‖y − x_d‖ / ‖x‖"); axs[0].set_title("Efecto relativo (mejor retardo entero)", fontsize=11)
axs[0].legend(fontsize=9, loc="lower right")
axs[1].plot(cds, [r["R"] for r in e2], "o-", color=FIG["orange"], lw=1.8)
axs[1].set_ylabel("R [muestras]"); axs[1].set_title("Largo del chirp R a 72 GS/s", fontsize=11)
Lh = 512
n = np.arange(-Lh // 2, Lh // 2)
for cd, col in ((17, FIG["aqua"]), (200, FIG["blue"]), (800, FIG["orange"])):
    h = np.fft.fftshift(np.fft.ifft(cd_transfer(Lh, u.FS, D_si(u.D_PS), u.LAM, u.z_m(cd))))
    R = u.cd_R_of(cd)
    axs[2].semilogy(n, np.abs(h), color=col, lw=1.4, label=f"{cd} ps/nm (R = {R})")
    for s in (-1, 1):
        axs[2].axvline(s * (R - 1) / 2, color=col, ls=":", lw=1.2)
axs[2].set_xlim(-80, 80); axs[2].set_ylim(1e-4, 1.5)
axs[2].set_xlabel("n [muestras]"); axs[2].set_ylabel("|h[n]|")
axs[2].set_title("Respuesta al impulso (líneas: ±(R−1)/2)", fontsize=11); axs[2].legend(fontsize=9)
fig.suptitle("G-E2 · Fuerza de la CD en el rango de distancias", fontweight="bold")
u.save_fig(fig, "G-E2_fuerza_cd"); plt.show()
TIMES["E2_s"] = time.perf_counter() - t

**Observaciones.**

- El efecto relativo pasa de 0,173 a 17 ps/nm (1 km) a 1,02 a 200 ps/nm y 1,26 a 800 ps/nm. Desde unos 150 ps/nm (1,00) la salida ya no se parece a la entrada desplazada.
- R vale 1 hasta 25 ps/nm, 9 a 200 ps/nm y 33 a 800 ps/nm, a 72 GS/s.
- En la respuesta al impulso, |h| no termina en ±(R − 1)/2: las colas siguen fuera de las líneas punteadas. Por eso un FIR de K = R coeficientes no alcanza para el error del método (ver E3).

## E3 · Arquitectura de la CD

**Decisión que respalda (propuesta):** overlap-save en frecuencia (C) con L = 256 y V = 64; la forma de primer orden (A) y el FIR en el tiempo (B) quedan descartados en este rango.

- **(A)** y = x + j(s∗x), s real y simétrico de K<sub>s</sub> = 5, 7, 9 coeficientes; extendida y = x + j(s∗x) + (c∗x) con K<sub>s</sub> = 9 y K<sub>c</sub> = 3 (`utils_08.fit_first_order`).
- **(B)** FIR complejo centrado de K impar hasta 81, mínimos cuadrados (mismo modelo que `utils_08.fit_fir_complex`, resuelto con la matriz de Gram).
- **(C)** overlap-save con (L, V) = (128, 32), (256, 64), (512, 128).

In [ ]:
t = time.perf_counter()
A_CFG = (("A K_s=5", 5, 0), ("A K_s=7", 7, 0), ("A K_s=9", 9, 0), ("A ext. K_s=9, K_c=3", 9, 3))
E3 = {}
for cd, S in CASES:
    r = {"R": u.cd_R_of(cd)}
    yx, yy = ref(cd, S)
    for lab, Ks, Kc in A_CFG:
        r[lab] = max(u8.fit_first_order(x, y, Ks, Kc, TR, TE)[2] for x, y in ((X, yx), (Y, yy)))
    cx, cy = u.fir_curve(X, yx, TR, TE), u.fir_curve(Y, yy, TR, TE)
    r["B_curve"] = {K: max(cx[K], cy[K]) for K in cx}
    for L, V in u.OS_PAIRS:
        r[f"C ({L},{V})"] = max(nrmse(u.os_aligned(x, L, V, cd, S)[TE], y[TE]) for x, y in ((X, yx), (Y, yy)))
    E3[(cd, S)] = r
TIMES["E3_calculo_s"] = time.perf_counter() - t
print(f"{len(E3)} casos en {TIMES['E3_calculo_s']:.0f} s")

In [ ]:
CD_POS = [cd for cd in u.CD_GRID if cd > 0]
OPTS_A = [lab for lab, _, _ in A_CFG]
OPTS_C = [f"C ({L},{V})" for L, V in u.OS_PAIRS]
fig, ax = plt.subplots(figsize=(11, 6.2))
ax.axvspan(CD_POS[0], u.CD_NOMINAL, color=FIG["band1"], alpha=0.6, lw=0, label="rango nominal (≤ 200)")
ax.axvspan(u.CD_NOMINAL, u.CD_OPTIONAL, color=FIG["band2"], alpha=0.6, lw=0, label="rango opcional (200–800)")
styA = dict(zip(OPTS_A, [(FIG["yellow"], "v"), (FIG["orange"], "s"), (FIG["red"], "D"), (FIG["violet"], "P")]))
styC = dict(zip(OPTS_C, ["#86b6ef", FIG["blue"], "#104281"]))
for lab in OPTS_A:
    c, mk = styA[lab]
    ax.loglog(CD_POS, [E3[(cd, 0.0)][lab] for cd in CD_POS], color=c, marker=mk, ms=6, lw=1.6, ls="--", label="(" + lab[0] + ")" + lab[1:])
for lab in OPTS_C:
    ax.loglog(CD_POS, [E3[(cd, 0.0)][lab] for cd in CD_POS], color=styC[lab], marker="o", ms=6, lw=2.2, label="(C)" + lab[1:])
for tol, txt in ((1e-2, "1e-2 (aceptación)"), (1e-3, "1e-3")):
    ax.axhline(tol, color=FIG["ink"], lw=1.0, ls=":")
    ax.text(55, tol * 1.12, txt, fontsize=9, color=FIG["ink"])
ax.set_xlabel("CD acumulada [ps/nm] (S = 0; CD = 0 en la tabla)"); ax.set_ylabel("NRMSE (peor de X e Y)")
ax.set_xticks(CD_POS, [str(c) for c in CD_POS], fontsize=8)
ax.minorticks_off()
fig.legend(loc="outside lower center", ncol=4, fontsize=9)
fig.suptitle("G-E3a · Error del método contra CD acumulada: (A) primer orden y (C) overlap-save", fontweight="bold")
u.save_fig(fig, "G-E3a_nrmse_vs_cd"); plt.show()

In [ ]:
t3b, t3_all = [], []
for (cd, S), r in E3.items():
    row = {"CD ps/nm": cd, "S": S, "R": r["R"]}
    for tol in u.TOLS_CD:
        row[f"K (B) {tol:g}"] = fmtK(u.min_K_tol(r["B_curve"], tol))
    for lab in OPTS_C:
        row[lab] = r[lab]
    t3b.append(row)
    t3_all.append({**{k: v for k, v in row.items()}, **{lab: r[lab] for lab in OPTS_A},
                   **{f"B K={K}": v for K, v in r["B_curve"].items()}})
show(t3b, d=3)
u.save_csv(t3b, "T-E3b_K_fir_y_error_os"); u.save_csv(t3_all, "T-E3_todas_las_opciones")

In [ ]:
def reach(err_of, tol):
    # mayor CD de la grilla (S = 0) hasta la que se cumple en todos los puntos desde 0
    best = None
    for cd in u.CD_GRID:
        if err_of(cd) <= tol:
            best = cd
        else:
            break
    return "ninguno" if best is None else best


B_err = lambda K: (lambda cd: E3[(cd, 0.0)]["B_curve"][K])
opts = [(lab, (lambda cd, lab=lab: E3[(cd, 0.0)][lab])) for lab in OPTS_A]
opts += [(f"B K ≤ {u.K_B_MAX}", lambda cd: min(E3[(cd, 0.0)]["B_curve"].values()))]
opts += [(lab, (lambda cd, lab=lab: E3[(cd, 0.0)][lab])) for lab in OPTS_C]
t3c = [{"opción": lab, "alcance 1e-2 [ps/nm]": reach(f, 1e-2), "alcance 1e-3 [ps/nm]": reach(f, 1e-3)} for lab, f in opts]
show(t3c)
u.save_csv(t3c, "T-E3c_alcance")

In [ ]:
Lc, Vc, CDc = 256, 64, 200
f, G, Hid = u.os_effective_response(Lc, Vc, CDc)
OVER = 16
err_rms = np.sqrt(np.mean(np.abs(G - Hid) ** 2, axis=0))  # |G_i − H| RMS sobre las posiciones
ph = np.degrees(np.angle(G / Hid))
ph_rms = np.sqrt(np.mean(ph**2, axis=0))
Wsig = u8.rc_weight(f)
pred = float(np.sqrt(np.sum(Wsig * err_rms**2) / np.sum(Wsig)))
# en los bins de la FFT el error es 0; se grafica el máximo en cada intervalo entre bins (envolvente)
order = np.argsort(f)
env = lambda v: v[order].reshape(-1, OVER).max(1)
fc = f[order].reshape(-1, OVER).mean(1) / u.RS
fig, axs = plt.subplots(1, 2, figsize=(14, 4.8), sharex=True)
axs[0].semilogy(fc, env(np.abs(G - Hid).max(0)), color="#86b6ef", lw=1.4, label="máximo sobre las posiciones")
axs[0].semilogy(fc, env(err_rms), color=FIG["blue"], lw=1.8, label="RMS sobre las posiciones")
axs[0].set_ylabel("|G_i(f) − H(f)|  (|H| = 1)")
axs[1].plot(fc, env(np.abs(ph).max(0)), color="#86b6ef", lw=1.4, label="máximo sobre las posiciones")
axs[1].plot(fc, env(ph_rms), color=FIG["blue"], lw=1.8, label="RMS sobre las posiciones")
axs[1].set_ylabel("|fase de G_i / H| [°]")
for ax in axs:
    for fx in (-0.55, 0.55):
        ax.axvline(fx, color=FIG["muted"], ls=":", lw=1.2)
    for fx in (-0.6, 0.6):
        ax.axvline(fx, color=FIG["ink"], ls="--", lw=1.0)
    ax.set_xlabel("f / Rs  (punteado: borde de la señal ±0,55·Rs; guiones: Nyquist ±0,6·Rs)")
axs[0].legend(fontsize=9, loc="upper center")
fig.suptitle(f"G-E3d · Error en frecuencia del overlap-save (L, V) = ({Lc}, {Vc}) contra el H ideal, {CDc} ps/nm "
             f"(envolvente entre bins; G_i: posición i del bloque)", fontweight="bold", fontsize=12)
u.save_fig(fig, "G-E3d_error_frecuencia_os"); plt.show()
inband = np.abs(f) <= 0.55 * u.RS
low = np.abs(f) <= 0.3 * u.RS
emax = np.abs(G - Hid).max(0)
print(f"|G_i − H| máx: |f| ≤ 0,3·Rs {emax[low].max():.2e}; |f| ≤ 0,55·Rs {emax[inband].max():.2e}; fuera de banda {emax[~inband].max():.2e}")
print(f"|fase| máx en banda: {np.abs(ph[:, inband]).max():.3g}°; fuera de banda: {np.abs(ph[:, ~inband]).max():.3g}°")
print(f"NRMSE predicho desde el espectro (peso |H_RRC|²): {pred:.3e}  |  medido en E3: {E3[(CDc, 0.0)][f'C ({Lc},{Vc})']:.3e}")
TIMES["E3_s"] = time.perf_counter() - t

**Observaciones.**

- **T-E3c**, alcance con NRMSE ≤ 1e-2:
  - (A) llega a 10 ps/nm con K<sub>s</sub> = 5, 7 o 9, y la extendida a 17 ps/nm. Con 1e-3, (A) solo cumple en CD = 0 y la extendida en 5 ps/nm. En el rango nominal, (A) llega a NRMSE 1,1 a 200 ps/nm (G-E3a).
  - (C) con (128, 32) llega a 400 ps/nm (1,1e-2 a 600). Con (256, 64) cubre los 800 ps/nm con 1e-2 y llega a 300 ps/nm con 1e-3 (1,17e-3 a 400). Con (512, 128) cubre los 800 ps/nm con 1e-3.
- **T-E3b:**
  - el FIR (B) necesita K = 15 / 27 (1e-2 / 1e-3) a 200 ps/nm y K = 39 / 59 a 800 ps/nm;
  - (C) con (256, 64) da 5,70e-4 a 200 ps/nm y 2,65e-3 a 800 ps/nm;
  - con S = 0,057 los errores casi no cambian: 5,81e-4 y 2,69e-3 con (C) (256, 64), y los mismos K para (B).
- **G-E3d:**
  - el error de (C) está en los bordes de la banda: |G<sub>i</sub> − H| ≤ 1,9e-3 para |f| ≤ 0,3·R<sub>s</sub>, ≤ 9,2e-3 hasta el borde de la señal (0,55·R<sub>s</sub>) y hasta 6,3e-2 entre el borde y Nyquist;
  - la fase se aparta ≤ 0,53° en banda;
  - el NRMSE predicho desde el espectro (6,0e-4) coincide con el medido sobre la señal (5,7e-4).

## E4 · Recursos (estimación por fórmula, no síntesis)

**Respalda:** la elección de (C) por recursos y el paralelismo de referencia P = 48 muestras por ciclo por canal.

**Todo lo de esta sección es estimación por fórmula** (o valor externo citado), no síntesis. 2 polarizaciones; dispositivo con 4272 DSP.
- **(A)** primer orden, 2 productos de 8 bits por DSP: DSP = 2·P·K<sub>u</sub>, K<sub>u</sub> = (K<sub>s</sub> + 1)/2.
- **(B)** FIR complejo simétrico (S = 0): DSP = 2·P·⌈K/2⌉·3, con K el menor de E3 para cada tolerancia.
- **(C)** (L, V) = (256, 64): **valores de un análisis externo** (conteo de rotadores por carril, rotadores constantes en LUT y variables en DSP, 3 DSP por multiplicador complejo), no recalculados:

| P | DSP | LUT |
|---|---|---|
| 48 | 1128 | ~125 000 a 145 000 |
| 96 | 2256 | ~250 000 a 290 000 |
| 192 | 1536 | ~640 000 a 790 000 (no entra en LUT) |

In [ ]:
t = time.perf_counter()
C_EXT = {48: 1128, 96: 2256, 192: 1536}
dspA = lambda P, Ks: 2 * P * (Ks + 1) // 2
dspB = lambda P, K: None if K is None else 2 * P * (-(-K // 2)) * 3
e4 = []
for cd in u.CD_GRID:
    r = E3[(cd, 0.0)]
    row = {"CD ps/nm": cd}
    for P in u.PS:
        for tol in u.TOLS_CD:
            K = u.min_K_tol(r["B_curve"], tol)
            row[f"B {tol:g} P={P}"] = dspB(P, K) if K else f">{dspB(P, u.K_B_MAX)}"
    e4.append(row)
e4_fixed = [{"opción": f"(A) K_s = {Ks}", **{f"P={P}": dspA(P, Ks) for P in u.PS}} for Ks in (5, 7, 9)]
e4_fixed.append({"opción": "(C) (256, 64), externo", **{f"P={P}": C_EXT[P] for P in u.PS}})
e4_fixed.append({"opción": "% de 4272 para (C)", **{f"P={P}": round(100 * C_EXT[P] / u.DSP_TOTAL, 1) for P in u.PS}})
display(Markdown("**DSP de (B) por CD, P y tolerancia (estimación por fórmula)**"))
show(e4)
display(Markdown("**DSP de (A) y (C), independientes de la CD (estimación por fórmula; (C) externo)**"))
show(e4_fixed)
u.save_csv(e4, "T-E4_dsp_fir_por_cd"); u.save_csv(e4_fixed, "T-E4_dsp_primer_orden_y_os")

P0 = 48
fig, ax = plt.subplots(figsize=(10.5, 5.6))
ax.axvspan(0, u.CD_NOMINAL, color=FIG["band1"], alpha=0.6, lw=0, label="rango nominal (≤ 200)")
for tol, col, mk in ((1e-2, FIG["orange"], "s"), (1e-3, FIG["red"], "D")):
    xs, ys = [], []
    for cd in u.CD_GRID:
        K = u.min_K_tol(E3[(cd, 0.0)]["B_curve"], tol)
        if K:
            xs.append(cd); ys.append(dspB(P0, K))
    ax.plot(xs, ys, color=col, marker=mk, lw=1.8, label=f"(B) FIR, NRMSE ≤ {tol:g}")
okC = [cd for cd in u.CD_GRID if E3[(cd, 0.0)]["C (256,64)"] <= 1e-2]
ax.plot(okC, [C_EXT[P0]] * len(okC), color=FIG["blue"], marker="o", lw=2.2, label="(C) (256, 64), externo, puntos con NRMSE ≤ 1e-2")
ax.axhline(u.DSP_TOTAL, color=FIG["ink"], lw=1.2, ls="--")
ax.text(10, u.DSP_TOTAL * 1.02, "4272 DSP del dispositivo", fontsize=9, color=FIG["ink"])
ax.set_xlabel("CD acumulada [ps/nm]"); ax.set_ylabel("DSP"); ax.set_xlim(0, 820)
ax.legend(fontsize=9, loc="upper left")
fig.suptitle(f"G-E4 · DSP contra CD acumulada con P = {P0} (estimación por fórmula, no síntesis)", fontweight="bold")
u.save_fig(fig, "G-E4_dsp_vs_cd_P48"); plt.show()
TIMES["E4_s"] = time.perf_counter() - t

**Nota sobre (C) con P = 192** (análisis externo): con P = 192 la FFT de 256 es totalmente paralela (un bloque por ciclo): todos los twiddles son constantes y van en LUT, y solo × H̄ usa DSP (768 por polarización). Con P = 96 (un bloque cada 2 ciclos), los rotadores de la primera etapa alternan entre dos valores y van en DSP. Por eso P = 192 usa menos DSP que P = 96 pero no entra en LUT (~150 a 186 %).

**Observaciones.**

- **P = 48** (estimación por fórmula):
  - (B) necesita 2304 DSP (1e-2) y 4032 DSP (1e-3) a 200 ps/nm, y 5760 / 8640 DSP a 800 ps/nm, más que los 4272 del dispositivo;
  - (C) (256, 64), dato externo, usa 1128 DSP (26,4 %) en todo el rango;
  - (A) usa 288 a 480 DSP, pero solo llega a 10 ps/nm (E3).
- **P = 96:** (B) ya pide 4608 DSP a 200 ps/nm con 1e-2, por encima de 4272, y (C) usa 2256 DSP (52,8 %).
- **P = 192:** (B) supera 4272 desde 25 ps/nm (4608 DSP con 1e-2). (C) usa 1536 DSP (36 %), pero no entra en LUT (dato externo, nota de arriba).

## E5 · Rango en Q1.15 y nivel de la señal

**Decisión que respalda (propuesta, a confirmar con el equipo):** con la CD hasta 200 a 800 ps/nm, 1 bit entero no alcanza con RMS 0,25; se propone bajar el RMS común.

- **RMS de entrada:** 0,25; 0,22; 0,20 y 0,18, con un factor común a los 4 canales que se recalcula en cada corrida.
- **Corridas:** los 4 desfasajes de la señal común (0, ⌊P/4⌋, ⌊P/2⌋, ⌊3P/4⌋ bits, P = 32 767) se corren juntos r·⌊P/16⌋ = r·2047 bits, r = 0 a 3 (r = 0 es la señal del 08). Se informa el peor caso de las 4 corridas y de los 4 canales: pico máximo, saturaciones máximas, SQNR mínima. Sin 1024 muestras por borde.
- **Señales:** entrada; CD ideal (`cd_reference`) en toda la grilla y con S = 0,057 a 200 y 800; como control, la salida de (C) con (256, 64) a 200 y 800 ps/nm; salidas de la ISI (3 perfiles con m = 4, B = 0,3 a 0,5·R<sub>s</sub>, diseño (e) con K = 12, h tal como sale del diseño).
- **SQNR por saturación:** Q1.15 con saturación contra Q1.15 sin saturar (∞ si no satura). **SQNR de la cuantización sola:** redondeo a Q1.15 sin saturar.
- **Corrida larga** (aparte): r = 0 con 327 680 símbolos (4 veces más).

In [ ]:
t = time.perf_counter()
RMS_LIST = (0.25, 0.22, 0.20, 0.18)
SHIFT = u8.PERIOD // 16
RUNS = [u.signal_run(r * SHIFT) for r in range(4)]
assert np.allclose(RUNS[0]["channels"], CH)  # r = 0 es la señal común del 08
print("corridas: desfasajes (bits) por canal:", [run["offsets"] for run in RUNS])

# ISI: diseño (e) con K = 12 (no usa datos; mismo cálculo que el criterio (e) del 08)
TR3, TE3 = u8.split_indices(N, 512)
GRAM = u8.LagGram(X, TR3)
W_RC = u8.rc_weight(GRAM.f_band)
G_RC = GRAM.G_weighted(W_RC)
H12 = {}
for kind, m in M4:
    for B in (0.3, 0.4, 0.5):
        H12[(kind, m, B)] = u.design_e(GRAM, G_RC, W_RC, kind, m, B, 12, X, TE3)


def sources(chans):
    # (fuente, CD, S) -> 4 canales reales a RMS 0,25
    x = chans[0] + 1j * chans[1]; y = chans[2] + 1j * chans[3]
    out = {("entrada", None, None): chans}
    for cd, S in CASES:
        yx, yy = u.cd_ref(x, cd, S), u.cd_ref(y, cd, S)
        out[("CD ideal", cd, S)] = np.array([yx.real, yx.imag, yy.real, yy.imag])
    for cd in (200, 800):
        yx, yy = u.os_aligned(x, 256, 64, cd), u.os_aligned(y, 256, 64, cd)
        out[("CD (C) (256,64)", cd, 0.0)] = np.array([yx.real, yx.imag, yy.real, yy.imag])
    for (kind, m, B), (h, start, *_r) in H12.items():
        out[(f"ISI {u8.profile_name(kind, m)} B={B}", None, None)] = u.apply_real_fir(chans, h, start)
    return out


E5 = {}  # (fuente, CD, S, RMS) -> lista de stats por corrida
for run in RUNS:
    for key, chans in sources(run["channels"]).items():
        for rms in RMS_LIST:
            E5.setdefault(key + (rms,), []).append(u.stats4(chans * rms / u8.RMS_TARGET, INNER))
TIMES["E5_corridas_s"] = time.perf_counter() - t
print(f"{len(E5)} combinaciones × {len(RUNS)} corridas en {TIMES['E5_corridas_s']:.0f} s")

In [ ]:
def worst(lst):
    return {"pico": max(s["peak_max"] for s in lst), "n_sat": max(s["n_sat_total"] for s in lst),
            "SQNR sat dB": min(s["sqnr_sat_min_db"] for s in lst), "SQNR total dB": min(s["sqnr_full_min_db"] for s in lst),
            "pico por corrida": " / ".join(f"{s['peak_max']:.4f}" for s in lst)}


e5_long = []
for (src, cd, S, rms), lst in E5.items():
    e5_long.append({"fuente": src, "CD ps/nm": cd, "S": S, "RMS": rms, **worst(lst)})
u.save_csv(e5_long, "T-E5_rango_q115_todas")

# tabla compacta: pico (n_sat) por RMS
def compact(sel):
    rows = []
    keys = list(dict.fromkeys((r["fuente"], r["CD ps/nm"], r["S"]) for r in e5_long if sel(r)))
    for k in keys:
        row = {"fuente": k[0], "CD ps/nm": "" if k[1] is None else k[1], "S": "" if k[2] is None else k[2]}
        for rms in RMS_LIST:
            r = next(r for r in e5_long if (r["fuente"], r["CD ps/nm"], r["S"]) == k and r["RMS"] == rms)
            sq = "∞" if np.isinf(r["SQNR sat dB"]) else f"{r['SQNR sat dB']:.1f}"
            row[f"RMS {rms}: pico / n_sat / SQNR sat"] = f"{r['pico']:.4f} / {r['n_sat']} / {sq}"
        rows.append(row)
    return rows


e5_cd = compact(lambda r: r["fuente"] in ("entrada", "CD ideal", "CD (C) (256,64)"))
e5_isi = compact(lambda r: r["fuente"].startswith("ISI"))
display(Markdown("**Entrada y CD: peor caso de 4 corridas × 4 canales (pico / muestras saturadas / SQNR por saturación en dB)**"))
show(e5_cd)
display(Markdown("**ISI, diseño (e), K = 12: peor caso de 4 corridas × 4 canales**"))
show(e5_isi)
u.save_csv(e5_cd, "T-E5_cd_peor_caso"); u.save_csv(e5_isi, "T-E5_isi_peor_caso")

In [ ]:
qs = [{"RMS": rms, "SQNR cuantización sola, entrada [dB] (mín. 4 canales)":
       min(u.quant_only_sqnr_db(v[INNER] * rms / u8.RMS_TARGET) for v in CH),
       "costo contra 0,25 [dB]": None} for rms in RMS_LIST]
for r in qs:
    r["costo contra 0,25 [dB]"] = qs[0]["SQNR cuantización sola, entrada [dB] (mín. 4 canales)"] - r["SQNR cuantización sola, entrada [dB] (mín. 4 canales)"]
show(qs, d=4)
u.save_csv(qs, "T-E5_sqnr_cuantizacion_sola")


def max_rms_ok(cd_max):
    # mayor RMS sin saturación en entrada, CD ideal (S = 0 y 0,057) hasta cd_max e ISI
    ok = []
    for rms in RMS_LIST:
        sel = [r for r in e5_long if r["RMS"] == rms and (
            r["fuente"] == "entrada" or r["fuente"].startswith("ISI")
            or (r["fuente"] == "CD ideal" and r["CD ps/nm"] <= cd_max))]
        if all(r["n_sat"] == 0 for r in sel):
            ok.append(rms)
    return max(ok) if ok else "ninguno de la lista"


RMS_NOM, RMS_OPT = max_rms_ok(u.CD_NOMINAL), max_rms_ok(u.CD_OPTIONAL)
print(f"mayor RMS sin saturación, rango nominal (≤ 200 ps/nm): {RMS_NOM}")
print(f"mayor RMS sin saturación, rango opcional (≤ 800 ps/nm): {RMS_OPT}")

In [ ]:
t = time.perf_counter()
LONG = u.signal_run(0, n_sym=4 * u8.N_SYM)
CHL = LONG["channels"]
NL = CHL.shape[1]
INL = slice(EDGE, NL - EDGE)
XL, YL = CHL[0] + 1j * CHL[1], CHL[2] + 1j * CHL[3]
e5L = []
for cd, S in [(None, None)] + CASES:
    if cd is None:
        ch = CHL
        key = ("entrada", None, None)
    else:
        yx, yy = u.cd_ref(XL, cd, S), u.cd_ref(YL, cd, S)
        ch = np.array([yx.real, yx.imag, yy.real, yy.imag])
        key = ("CD ideal", cd, S)
    for rms in (0.25, 0.20):
        s = u.stats4(ch * rms / u8.RMS_TARGET, INL)
        s0 = E5[key + (rms,)][0]
        e5L.append({"fuente": key[0], "CD ps/nm": cd, "S": S, "RMS": rms, "pico r=0": s0["peak_max"],
                    "pico largo (×4)": s["peak_max"], "Δ pico": s["peak_max"] - s0["peak_max"],
                    "n_sat r=0": s0["n_sat_total"], "n_sat largo": s["n_sat_total"]})
display(Markdown(f"**Corrida larga: {4 * u8.N_SYM} símbolos ({NL} muestras por canal) contra r = 0 ({N} muestras); RMS 0,25 y 0,20**"))
show(e5L)
u.save_csv(e5L, "T-E5_corrida_larga")
TIMES["E5_larga_s"] = time.perf_counter() - t

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 5.6))
ax.axvspan(0, u.CD_NOMINAL, color=FIG["band1"], alpha=0.6, lw=0, label="rango nominal (≤ 200)")
ax.axvspan(u.CD_NOMINAL, u.CD_OPTIONAL, color=FIG["band2"], alpha=0.6, lw=0, label="rango opcional (200–800)")
for rms, col in zip(RMS_LIST, ("#104281", "#2a78d6", "#5598e7", "#86b6ef")):
    pk = [max(s["peak_max"] for s in E5[("CD ideal", cd, 0.0, rms)]) for cd in u.CD_GRID]
    ax.plot(u.CD_GRID, pk, marker="o", color=col, lw=1.8, label=f"RMS {rms}")
ax.axhline(1.0, color=FIG["red"], lw=1.3, ls="--")
ax.text(230, 1.02, "fondo de escala de Q1.15 (1 bit entero)", fontsize=9, color=FIG["red"])
ax.set_xlabel("CD acumulada [ps/nm] (S = 0)"); ax.set_ylabel("pico de salida (peor de 4 corridas × 4 canales)")
ax.set_xlim(0, 820)
ax.legend(fontsize=9, loc="lower right", ncol=2)
fig.suptitle("G-E5 · Pico de salida de la CD ideal contra CD acumulada, por RMS de entrada", fontweight="bold")
u.save_fig(fig, "G-E5_pico_cd_vs_rms"); plt.show()
TIMES["E5_s"] = time.perf_counter() - t + TIMES["E5_corridas_s"]

**Observaciones.**

- **Con RMS 0,25 la CD satura dentro del rango nominal:**
  - a 200 ps/nm el pico llega a 1,1196 con 2 muestras saturadas y SQNR por saturación de 55,3 dB, por debajo de 60 dB;
  - a 800 ps/nm, el pico es 1,1791, con 22 muestras saturadas y 50,1 dB;
  - la salida de (C) (256, 64) da el mismo pico (1,1197 y 1,1792), así que la saturación es de la señal y no del método.
- **Mayor RMS de la lista sin saturar:**
  - **0,22 en el rango nominal**: pico 0,9852 a 200 ps/nm. Con 0,22, a 300, 600 y 800 ps/nm satura 1 o 2 muestras (SQNR por saturación de 65,2 a 80,9 dB).
  - **0,20 en el opcional**: pico 0,9433 a 800 ps/nm.
  - Bajar de 0,25 a 0,20 cuesta 1,95 dB de SQNR de cuantización (89,05 → 87,10 dB), y a 0,18 cuesta 2,84 dB.
  - La entrada (pico 0,6985) y las salidas de la ISI (pico máximo 0,7214, Butterworth m = 4, B = 0,5) no saturan con ningún RMS.
- **El pico depende de qué muestras se tomen.**
  - A 200 ps/nm con RMS 0,25, las 4 corridas dan 1,0702 / 1,0600 / 1,1190 / 1,1196.
  - La corrida larga (×4 símbolos) sube de 1,070 a 1,119 y de 2 a 6 muestras saturadas, pero no supera el peor de las 4 corridas cortas. A 800 ps/nm la corrida larga da 1,179, contra 1,178 de r = 0.
  - Con una sola PRBS15, más símbolos repiten la misma secuencia m (período de 32 767 símbolos).

## E6 · ISI

**Decisión que respalda (propuesta):** 3 perfiles con m = 4 (gaussiano, Bessel, Butterworth), B ≤ 0,5·R<sub>s</sub>, diseño por mínimos cuadrados pesado por |H<sub>RRC</sub>|² (criterio (e)), K = 12, NRMSE ≤ 1e-2, acumulador con 2 bits enteros.

Se reutilizan T5, T5b, T6 y T7 del 08 (`resultados/08_escenario_60gbd/csv/`) y solo se calcula lo que falta: el criterio (e) en B = 0,6·R<sub>s</sub>, y los h de (e) con K = 12 (los mismos de E5) para las figuras.

In [ ]:
t = time.perf_counter()
T5 = u.read_csv_08("T5_isi_K_minimo")
CURV = u.read_csv_08("T5_isi_curvas_vs_K")
T5B = u.read_csv_08("T5b_isi_acumulador")
NYQ = u.read_csv_08("T5_salto_en_nyquist")
T7 = u.read_csv_08("T7_dsp_isi_ffa")
# control: los h de (e) con K = 12 reproducen el NRMSE del 08
ctl = []
for (kind, m, B), (h, start, err, *_r) in H12.items():
    r08 = next(float(c["nrmse_e"]) for c in CURV if c["perfil"] == u8.profile_name(kind, m)
               and float(c["B_Rs"]) == B and int(c["K"]) == 12)
    ctl.append({"perfil": u8.profile_name(kind, m), "B_Rs": B, "NRMSE (e) K=12, 09": err, "NRMSE (e) K=12, 08": r08,
                "dif. relativa": abs(err - r08) / r08})
show(ctl, d=5)
assert max(r["dif. relativa"] for r in ctl) < 1e-6
# criterio (e) en B = 0,6 (falta en el 08)
E06 = []
for kind, m in M4:
    r = u8.isi_case(X, kind, m, 0.6, TR3, TE3, GRAM, weights={"lsw": (G_RC, W_RC)})
    row = {"perfil": u8.profile_name(kind, m), "B_Rs": 0.6}
    for tol in u8.TOLS:
        k = u8.min_K(r["lsw"], lambda e, tol=tol: e <= tol)
        row[f"e {tol:g}"] = ">64" if k is None else k
    row["NRMSE (e) K=12"] = r["lsw"][12]
    E06.append(row)
show(E06)
u.save_csv(E06, "T-E6a_criterio_e_B06")
TIMES["E6_calculo_s"] = time.perf_counter() - t

In [ ]:
num = lambda k: 68 if k in (">64", None) else int(k)
Bs = (0.3, 0.4, 0.5, 0.6)
PANELS = (("a (98,5 %)", "(a) energía ≥ 98,5 %"), ("b 0.01", "(b) truncado + ganancia, NRMSE ≤ 1e-2"),
          ("c 0.01", "(c) MC en frecuencia, NRMSE ≤ 1e-2"), ("e 0.01", "(e) MC pesado por |H_RRC|², NRMSE ≤ 1e-2"))
fig, axs = plt.subplots(1, 4, figsize=(17, 4.8), sharey=True)
for ax, (col, title) in zip(axs, PANELS):
    for kind, m in M4:
        name = u8.profile_name(kind, m)
        ks = []
        for B in Bs:
            if col.startswith("e") and B == 0.6:
                ks.append(num(next(r for r in E06 if r["perfil"] == name)["e 0.01"]))
            else:
                ks.append(num(next(r for r in T5 if r["perfil"] == name and float(r["B_Rs"]) == B)[col]))
        ax.plot(Bs, ks, marker="o", ms=7, lw=1.8, color=FAMILY[kind], label=name)
    ax.axhline(12, color=FIG["ink"], ls="--", lw=1.1)
    ax.text(0.305, 13, "K = 12", fontsize=9, color=FIG["ink"])
    ax.set_title(title, fontsize=10); ax.set_xticks(Bs); ax.set_xlabel("B / Rs")
axs[0].set_ylabel("K mínimo"); axs[0].set_ylim(0, 71)
axs[0].set_yticks([0, 12, 24, 36, 48, 64, 68], ["0", "12", "24", "36", "48", "64", ">64"])
h_, l_ = axs[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="outside lower center", ncol=3, fontsize=10)
fig.suptitle("G-E6a · Menor K contra B, perfiles con m = 4 (T5 del 08; (e) en B = 0,6 calculado acá)", fontweight="bold")
u.save_fig(fig, "G-E6a_isi_K_vs_B"); plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.6), sharey=True)
fpos = np.fft.rfftfreq(u8.L_REF, 1 / u.FS)
for ax, (kind, m) in zip(axs, M4):
    for B, col in zip(Bs, ("#86b6ef", "#3987e5", "#1c5cab", FIG["orange"])):
        H = np.abs(np.fft.rfft(u8.h_reference(kind, m, B)))
        ax.plot(fpos[:-1] / u.RS, 20 * np.log10(H[:-1] / H[0]), color=col, lw=1.8 if B == 0.6 else 1.4, label=f"B = {B}·Rs")
    ax.axvline(0.6, color=FIG["ink"], ls="--", lw=1.2)
    ax.axvline(0.55, color=FIG["muted"], ls=":", lw=1.2)
    ax.text(0.595, -1.5, "Nyquist", rotation=90, ha="right", va="top", fontsize=9, color=FIG["ink"])
    ax.text(0.545, -1.5, "borde señal", rotation=90, ha="right", va="top", fontsize=9, color=FIG["muted"])
    ax.set_title(u8.profile_name(kind, m), fontsize=11); ax.set_xlim(0, 0.62); ax.set_ylim(-12, 1)
    ax.set_xlabel("f / Rs")
axs[0].set_ylabel("|H_P(f)| / |H_P(0)| [dB]"); axs[0].legend(fontsize=9, loc="lower left")
fig.suptitle("G-E6b · Módulo del prototipo hasta Nyquist (0,6·Rs a 1,2 sps)", fontweight="bold")
u.save_fig(fig, "G-E6b_prototipo_nyquist"); plt.show()
nyq_m4 = [r for r in NYQ if r["perfil"] in {u8.profile_name(k, m) for k, m in M4}]
display(Markdown("**Salto en Nyquist y K de (b) y (d) con NRMSE ≤ 1e-2 (T5 del 08, sin recalcular)**"))
show(nyq_m4, d=3)

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
for j, (kind, m) in enumerate(M4):
    for B, col in ((0.4, FIG["blue"]), (0.5, FIG["orange"])):
        h, start, err, h_ref, o64 = H12[(kind, m, B)]
        g0 = abs(np.sum(h_ref))
        # prototipo rotado -o64 en su grilla de L_REF (la de su diseño): retardo chico, fase sin ambigüedad
        for hh, s, ls, lab in ((np.roll(h_ref, -o64), 0, "-", f"prototipo, B = {B}"), (h, start, "--", f"K = 12 (e), B = {B}")):
            Hh = np.fft.fft(hh, u8.L_REF)
            f, tau = group_delay(Hh, u.FS)
            mag = np.fft.fftshift(20 * np.log10(np.abs(Hh) / g0 + 1e-12))
            keep = (f >= 0) & (f <= 0.59 * u.RS)
            axs[0, j].plot(f[keep] / u.RS, mag[keep], color=col, ls=ls, lw=1.7, label=lab)
            axs[1, j].plot(f[keep] / u.RS, tau[keep] * u.FS + s, color=col, ls=ls, lw=1.7, label=lab)
    for ax in axs[:, j]:
        ax.axvline(0.55, color=FIG["muted"], ls=":", lw=1.2); ax.axvline(0.6, color=FIG["ink"], ls="--", lw=1.0)
    axs[0, j].set_title(u8.profile_name(kind, m), fontsize=11); axs[0, j].set_ylim(-12, 2)
    axs[1, j].set_xlabel("f / Rs  (punteado: borde de la señal; guiones: Nyquist)")
axs[0, 0].set_ylabel("|H| / |H_ref(0)| [dB]"); axs[1, 0].set_ylabel("τ [muestras desde o64] (hasta 0,59·Rs)")
h_, l_ = axs[0, 0].get_legend_handles_labels()
fig.legend(h_, l_, loc="outside lower center", ncol=4, fontsize=10)
fig.suptitle("G-E6c · Módulo y retardo de grupo: prototipo y h con K = 12, diseño (e)", fontweight="bold")
u.save_fig(fig, "G-E6c_modulo_retardo_K12"); plt.show()

In [ ]:
sym_x = SIG["levels"][0] + 1j * SIG["levels"][1]
sig_x = CommSignal(samples=X, symbols=sym_x, bits=np.zeros(0, np.uint8), scenario=u.ESC, seed=SIG["seeds"][0],
                   mod="16qam", prbs_order=u8.PRBS_ORDER, span=u8.SPAN_RX)
cases = [("sin ISI (back-to-back)", X)]
for kind, m in M4:
    h, start, *_r = H12[(kind, m, 0.4)]
    cases.append((f"{u8.profile_name(kind, m)}, B = 0,4·Rs, K = 12 (e)", u8.apply_at(X, h, start, np.arange(N))))
fig, axs = plt.subplots(1, 4, figsize=(17, 4.6))
e6d = []
for ax, (lab, xin) in zip(axs, cases):
    res = rx.recover(xin, sig_x)
    plots.constellation(ax, res.symbols[::4], res.ref)
    for ln in ax.lines:
        ln.set_rasterized(True)  # SVG liviano
    ax.set_title(f"{lab}\nEVM = {100 * res.evm:.2f} %", fontsize=10)
    ax.set_aspect("equal")
    e6d.append({"caso": lab, "EVM %": 100 * res.evm, "fase": res.phase, "retardo símb.": res.delay})
fig.suptitle("G-E6d · Constelación de X a la salida de la ISI (receptor a 1,2 sps de B5.4, filtro adaptado span 64)", fontweight="bold")
u.save_fig(fig, "G-E6d_constelacion_isi"); plt.show()
show(e6d, d=3)
u.save_csv(e6d, "T-E6d_evm")

In [ ]:
e6e = []
for K in (12, 16):
    for P in u.PS:
        c = u8.ffa_cost(K, P)
        dsp = int(round(4 * P * c["mult_per_sample"]))
        r08 = next((r for r in T7 if int(r["K"]) == K and int(r["P"]) == P), None)
        e6e.append({"K": K, "P": P, "a": c["a"], "b": c["b"], "K_p": c["K_p"], "mult/muestra": c["mult_per_sample"],
                    "DSP": dsp, "% de 4272": 100 * dsp / u.DSP_TOTAL,
                    "DSP en T7 del 08": None if r08 is None else int(r08["DSP"]),
                    "dif. con el 08": None if r08 is None else dsp - int(r08["DSP"])})
display(Markdown("**T-E6e · DSP de la ISI con FFA, 4 canales reales (estimación por fórmula, no síntesis)**"))
show(e6e)
u.save_csv(e6e, "T-E6e_dsp_isi")
t5b_e12 = [r for r in T5B if r["diseño"] == "e" and r["K"] == "12"]
display(Markdown("**T5b del 08 (sin recalcular): acumulador con diseño (e), K = 12**"))
show(t5b_e12, ["perfil", "B_Rs", "NRMSE", "Σ|h|", "cota", "pico salida", "pico parciales", "I cota", "I salida", "I parciales"])
TIMES["E6_s"] = time.perf_counter() - t

**Observaciones.**

- **K = 12 alcanza para B ≤ 0,5·R<sub>s</sub> con el diseño (e).**
  - En T5 del 08, el mayor K de (e) para NRMSE ≤ 1e-2 es 11 (Bessel m = 4, B = 0,5).
  - Con K = 12 el peor NRMSE es 7,13e-3. Este notebook lo reproduce con una diferencia relativa < 1e-6.
  - Con B = 0,6, (e) necesita K = 14 para Bessel m = 4 (NRMSE 1,22e-2 con K = 12). Gaussiano y Butterworth necesitan K = 4.
- **B = 0,6·R<sub>s</sub>, salto en Nyquist (T5 del 08):**
  - Bessel m = 4 salta 1,21 en Nyquist y (b) necesita K = 51.
  - Butterworth m = 4 no salta (fase 0 en f<sub>s</sub>/2) y (b) necesita K = 9.
  - El problema de B = 0,6 depende del perfil. A 1,2 sps, Nyquist (0,6·R<sub>s</sub>) coincide con B, así que el prototipo llega a f<sub>s</sub>/2 en −3 dB (G-E6b).
- **Efecto visible (G-E6d):** sin ecualizar, la EVM es 0,41 % sin ISI, 15,3 % con el gaussiano, 15,7 % con Bessel m = 4 y 24,1 % con Butterworth m = 4, todos con B = 0,4·R<sub>s</sub> y K = 12.
- **Recursos y acumulador:**
  - T-E6e coincide con T7 del 08 (diferencia 0): 864 / 1728 / 3456 DSP con K = 12 para P = 48 / 96 / 192.
  - Acumulador (T5b del 08, (e), K = 12): la cota pide 2 bits enteros en 6 de los 9 casos, y el pico real de las sumas parciales (0,7312) entra en 1 bit.

## E7 · Sensibilidad: emular la misma memoria del canal que el sistema real

**Hipótesis pendiente de decisión del equipo.** ¿El emulador tiene que reproducir los mismos km o la misma memoria de la CD en símbolos que el sistema real? La memoria en símbolos crece con D·z·R<sub>s</sub>², así que con el emulador a 60 GBd y el sistema real a R<sub>s,ref</sub> la CD acumulada equivalente es CD<sub>real</sub>·(R<sub>s,ref</sub>/60 GBd)².

Para R<sub>s,ref</sub> = 100, 200 y 400 GBd y 2, 10 y 40 km reales (D = 17): CD equivalente a 60 GBd, R a 72 GS/s y error de (C) con (L, V) = (256, 64), (512, 128), (1024, 256) y (2048, 512), medido como en E3. La última columna es solo una referencia, sin simular: el menor L potencia de 2 con V = L/4 ≥ R − 1.

In [ ]:
t = time.perf_counter()
e7 = []
for rs_ref in (100, 200, 400):
    for km in (2, 10, 40):
        cdeq = u.D_PS * km * (rs_ref / 60) ** 2
        yx, yy = u.cd_ref(X, cdeq), u.cd_ref(Y, cdeq)
        R = u.cd_R_of(cdeq)
        row = {"Rs_ref GBd": rs_ref, "km reales": km, "CD real ps/nm": u.D_PS * km, "CD equiv. ps/nm": cdeq, "R": R}
        first = None
        for L, V in u.OS_PAIRS_E7:
            e = max(nrmse(u.os_aligned(x, L, V, cdeq)[TE], y[TE]) for x, y in ((X, yx), (Y, yy)))
            row[f"({L},{V})"] = e
            if first is None and e <= 1e-2:
                first = f"({L},{V})"
        row["menor (L,V) con 1e-2"] = first or "ninguno de la lista"
        row["L mín. con L/4 ≥ R−1 (ref.)"] = u.min_L_pow2(R)
        e7.append(row)
display(Markdown("**T-E7 · hipótesis pendiente de decisión del equipo**"))
show(e7, d=3)
u.save_csv(e7, "T-E7_memoria_equivalente")
TIMES["E7_s"] = time.perf_counter() - t

**Observaciones.**

- Si el emulador tiene que reproducir la memoria en símbolos de un sistema más rápido, la CD equivalente sale del rango opcional: de 94 ps/nm (100 GBd, 2 km) a 30 222 ps/nm (400 GBd, 40 km, R = 1255).
- **(256, 64)** alcanza hasta 472 ps/nm (100 GBd y 10 km, o 200 GBd y 2 km). **(512, 128)** hace falta desde 1511 ps/nm (400 GBd y 2 km) y alcanza hasta 1889 ps/nm. **(2048, 512)** hace falta para 7556 ps/nm (200 GBd y 40 km, o 400 GBd y 10 km). Para 400 GBd y 40 km no alcanza ninguno de la lista: harían falta L = 8192 como referencia sin simular.
- La regla V = L/4 ≥ R − 1 es necesaria pero no suficiente. A 1511 ps/nm (R = 63), (256, 64) cumple la regla pero da 1,18e-2 > 1e-2. Por eso la columna de referencia no reemplaza a la simulación.

## E8 · Resumen de decisiones y evidencias

Los valores de la tabla salen de las celdas anteriores. **Estado:** "propuesta" = respaldada por este notebook, falta acordarla; "pendiente con el equipo" = necesita una decisión que este notebook no puede tomar.

In [ ]:
r200, r800 = E3[(200, 0.0)], E3[(800, 0.0)]
kb = lambda cd, tol: fmtK(u.min_K_tol(E3[(cd, 0.0)]["B_curve"], tol))
reachC = next(r for r in t3c if r["opción"] == "C (256,64)")
reachA = {r["opción"]: r["alcance 1e-2 [ps/nm]"] for r in t3c if r["opción"].startswith("A")}
pk = lambda rms, cd: max(s["peak_max"] for s in E5[("CD ideal", cd, 0.0, rms)])
kmax_e = max(int(r["e 0.01"]) for r in T5 if r["perfil"] in {u8.profile_name(k, m) for k, m in M4} and float(r["B_Rs"]) <= 0.5)
nrm12 = max(c["NRMSE (e) K=12, 09"] for c in ctl)
Ib = max(int(r["I cota"]) for r in t5b_e12); Ir = max(int(r["I parciales"]) for r in t5b_e12)
dB48 = lambda cd, tol: dspB(48, u.min_K_tol(E3[(cd, 0.0)]["B_curve"], tol))
none_e7 = [f"{r['Rs_ref GBd']} GBd, {r['km reales']} km" for r in e7 if r["menor (L,V) con 1e-2"] == "ninguno de la lista"]
fmt_rms = lambda v: v if isinstance(v, str) else f"{v:g}"
E8 = [
    ("Rango de CD", "0 a 200 ps/nm nominal; opcional hasta 800 ps/nm",
     f"E1 (fuentes externas [1] a [3]); G-E2: R = {u.cd_R_of(200)} a 200 y {u.cd_R_of(800)} a 800 ps/nm", "propuesta"),
    ("Parámetros de configuración de la CD", "D, S, z y λ; se informa también D·z",
     f"T-E1: mismo D·z da la misma salida (NRMSE ≤ {max(r['NRMSE contra D=17, z=200/17 km'] for r in e1[:3]):.1e}); "
     f"λ = 1310 nm cambia la salida (NRMSE {e1[3]['NRMSE contra D=17, z=200/17 km']:.2f}); S = 0,057 a 800 ps/nm: NRMSE {e1[-1]['NRMSE contra S = 0, mismo D·z']:.1e}", "propuesta"),
    ("Arquitectura de la CD", "(C) overlap-save, L = 256, V = 64",
     f"G-E3a, T-E3b, T-E3c, G-E3d: NRMSE {r200['C (256,64)']:.1e} a 200 y {r800['C (256,64)']:.1e} a 800 ps/nm; alcance 1e-2: {reachC['alcance 1e-2 [ps/nm]']} ps/nm", "propuesta"),
    ("Por qué no (A)", "descartada en el rango",
     f"T-E3c: alcance 1e-2 de (A) " + ", ".join(f"{k[2:]}: {v}" for k, v in reachA.items()) + " ps/nm", "propuesta"),
    ("Por qué no (B)", "descartada en el rango",
     f"T-E3b y G-E4: K = {kb(200, 1e-2)} / {kb(200, 1e-3)} a 200 ps/nm y {kb(800, 1e-2)} / {kb(800, 1e-3)} a 800 (1e-2 / 1e-3); "
     f"con P = 48, {dB48(200, 1e-2)} / {dB48(800, 1e-2)} DSP (1e-2, 200 / 800) contra 1128 de (C), externo", "propuesta"),
    ("Nivel de señal y bits enteros", f"1 bit entero (Q1.15) con RMS ≤ {fmt_rms(RMS_NOM)} (nominal) / ≤ {fmt_rms(RMS_OPT)} (opcional)",
     f"T-E5, G-E5: pico con RMS 0,25: {pk(0.25, 200):.3f} a 200 y {pk(0.25, 800):.3f} a 800 ps/nm; costo de bajar a 0,18: {qs[-1]['costo contra 0,25 [dB]']:.2f} dB de SQNR de cuantización", "pendiente con el equipo (RMS común)"),
    ("ISI: perfiles y B", "gaussiano, Bessel m = 4, Butterworth m = 4; B ≤ 0,5·Rs",
     "G-E6a, G-E6b (salto en Nyquist con B = 0,6), T5 del 08", "propuesta"),
    ("ISI: diseño y K", "(e) MC pesado por |H_RRC|², K = 12",
     f"G-E6a, G-E6c, T5 del 08: K máx. para 1e-2 con (e) y B ≤ 0,5 = {kmax_e}; NRMSE máx. con K = 12: {nrm12:.2e}", "propuesta"),
    ("ISI: acumulador", "2 bits enteros",
     f"T5b del 08: cota pide I = {Ib}, pico real I = {Ir} (diseño (e), K = 12)", "propuesta"),
    ("Paralelismo de referencia", "P = 48 por ciclo y por canal",
     f"G-E4, T-E4, T-E6e: (C) 1128 DSP con P = 48; con P = 192 no entra en LUT (externo); ISI K = 12: {next(r['DSP'] for r in e6e if r['K'] == 12 and r['P'] == 48)} DSP con P = 48", "propuesta"),
    ("Pendiente: km o memoria equivalente", "sin decidir",
     f"T-E7: sin par de la lista para {', '.join(none_e7) if none_e7 else '—'}", "pendiente con el equipo"),
    ("Pendiente: correlación de las PRBS15", "sin decidir", "08 (señal común) y guía §6; E5: corridas desplazadas y corrida larga", "pendiente con el equipo"),
    ("Pendiente: RMS común", "sin decidir", "T-E5 y G-E5", "pendiente con el equipo"),
]
e8 = [{"decisión": a, "valor propuesto": b, "evidencia": c, "estado": d} for a, b, c, d in E8]
show(e8)
u.save_csv(e8, "T-E8_resumen")

In [ ]:
meta = {
    "tarea": "D2.1",
    "notebook": "09_evidencias_cd_isi_72gsps.ipynb",
    "fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git": u8.git_info(),
    "versiones": {"numpy": np.__version__, "scipy": scipy.__version__, "matplotlib": plt.matplotlib.__version__},
    "escenario": u.ESC.to_dict(),
    "senal_comun": "utils_08.common_signal (ver meta.json del 08)",
    "cd": {"D_ps_nm_km": u.D_PS, "lambda_m": u.LAM, "grilla_ps_nm": list(u.CD_GRID), "S_alt_ps_nm2_km": u.S_ALT,
           "S_alt_en": list(u.CD_S_REPEAT), "nominal_ps_nm": u.CD_NOMINAL, "opcional_ps_nm": u.CD_OPTIONAL,
           "pares_os": [list(p) for p in u.OS_PAIRS], "pares_os_E7": [list(p) for p in u.OS_PAIRS_E7],
           "K_B_max": u.K_B_MAX, "tolerancias": list(u.TOLS_CD),
           "medicion": "contra cd_reference con el mismo S, segunda mitad sin 256 muestras por borde, peor de X e Y; (C) con retardo V/2"},
    "e5": {"rms": list(RMS_LIST), "desfasaje_bits": SHIFT, "corridas": [run["offsets"] for run in RUNS],
           "simbolos_corrida_larga": 4 * u8.N_SYM, "borde_muestras": EDGE,
           "rms_max_sin_saturar": {"nominal": RMS_NOM, "opcional": RMS_OPT}},
    "e4_externo": {"C_256_64_dsp": C_EXT, "nota": "analisis externo, no recalculado"},
    "e6": {"diseno": "e", "K": 12, "perfiles": [u8.profile_name(k, m) for k, m in M4], "reutiliza": "csv del 08"},
    "tiempos_s": {k: round(v, 1) for k, v in TIMES.items()} | {"total": round(time.perf_counter() - T0, 1)},
}
u.save_meta(meta)
print(meta["tiempos_s"])
print("guardado en", u.OUT)

## Cómo reproducir

```bash
cd models/floating_point/emulador-golden && . .venv/bin/activate
jupyter nbconvert --to notebook --execute --inplace notebooks/09_evidencias_cd_isi_72gsps.ipynb
jupyter nbconvert --to html notebooks/09_evidencias_cd_isi_72gsps.ipynb \
    --output-dir docs/golden_model/resultados/09_evidencias_cd_isi --output informe
```